<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [12]</a>'.</span>

# TorchLens in 10 minutes

Trace an eager PyTorch model, intervene on activations and observe the effect, run a backward pass and visualize it, see actual control flow rendered as IF / THEN / ELSE in the graph, and compare two traces as a Bundle.

Five features. CPU-only. Deterministic. About one minute end-to-end.

In [ ]:
from __future__ import annotations

import shutil
import tempfile
from pathlib import Path

from IPython.display import SVG, display
import torch
from torch import nn
import torch.nn.functional as F

import torchlens as tl

torch.manual_seed(0)
OUT = Path(tempfile.mkdtemp(prefix="torchlens_demo_"))
print(f"torchlens {tl.__version__}; tmp={OUT.name}")

In [ ]:
import sys

# Models live in a sidecar .py so TorchLens can read source for conditional
# branch labeling (the AST inspector needs a real file path).
sys.path.insert(0, str(Path.cwd().resolve()))
sys.path.insert(0, str((Path.cwd() / "notebooks").resolve()))

from _demo_models import TinyBranchCNN, TinyMLP

print("models:", TinyBranchCNN.__name__, "+", TinyMLP.__name__)

## 1. Trace a model

`tl.trace(model, x)` runs a real forward pass and records every operation, every saved tensor, and the full graph. The returned `Trace` exposes layers, ops, and modules as lookup-able accessors.

In [ ]:
cnn = TinyBranchCNN().eval()
x_pos = torch.full((1, 1, 8, 8), 1.0)
trace = tl.trace(cnn, x_pos, vis_opt="none")

# The label suffix is the global op-counter, so it shifts with model topology;
# pick the first conv2d to stay robust across edits.
conv_label = next(label for label in trace.layer_labels if label.startswith("conv2d"))
conv = trace.layers[conv_label]
{
    "first_layers": trace.layer_labels[:6],
    "conv_label": conv_label,
    "conv_out_shape": tuple(conv.out.shape),
    "conv_out_dtype": str(conv.out.dtype),
    "conv_module": conv.module,
    "num_ops": trace.num_ops,
    "num_modules": len(trace.modules),
    "num_params": trace.num_params,
    "total_flops": trace.total_flops,
}

## 2. Intervene and view the effect

Discover sites with selectors, fork the trace, attach a transform, replay, then compare outputs. The clean baseline stays untouched.

In [ ]:
mlp = TinyMLP().eval()
x_vec = torch.randn(2, 8)

clean = tl.trace(mlp, x_vec, intervention_ready=True, vis_opt="none")
zero_relu = clean.fork("zero_relu")
zero_relu.do(tl.func("relu"), tl.zero_ablate(), confirm_mutation=True)

clean_out = clean[clean.output_layers[0]].out
ablated_out = zero_relu[zero_relu.output_layers[0]].out

{
    "clean_norm": round(float(torch.linalg.vector_norm(clean_out)), 4),
    "zero_relu_norm": round(float(torch.linalg.vector_norm(ablated_out)), 4),
    "delta_norm": round(float(torch.linalg.vector_norm(clean_out - ablated_out)), 4),
    "clean_first_row": clean_out[0].tolist(),
    "zero_relu_first_row": ablated_out[0].tolist(),
}

## 3. Run a backward pass and visualize it

Capture saved grads with `save_grads=True`, then call `trace.log_backward(loss)`. The autograd graph is recorded with full forward / backward cross-references and can be rendered just like the forward graph.

In [ ]:
bwd_trace = tl.trace(mlp, x_vec, layers_to_save="all", save_grads=True, vis_opt="none")
loss = bwd_trace[bwd_trace.output_layers[0]].out.sum()
bwd_trace.log_backward(loss)

first_grad_label = bwd_trace.ops_with_saved_grads[0]
{
    "has_grads": bwd_trace.has_grads,
    "num_grad_fns": len(bwd_trace.grad_fns),
    "layers_with_saved_grads": bwd_trace.ops_with_saved_grads[:5],
    "first_grad_shape": tuple(bwd_trace[first_grad_label].grad.shape),
    "first_grad_l2": round(float(torch.linalg.vector_norm(bwd_trace[first_grad_label].grad)), 4),
}

In [ ]:
bwd_path = OUT / "backward"
bwd_trace.draw_backward(vis_outpath=str(bwd_path), vis_save_only=True, vis_fileformat="svg")
display(SVG(filename=str(bwd_path.with_suffix(".svg"))))

## 4. Conditional flow: see IF / THEN / ELSE in the graph

TorchLens captures the actual executed Python control flow. Every if-chain becomes a `Conditional` record with arms, source location, and the bool that decided the branch. The graph renderer labels arm-entry edges so you can see which path fired.

Two traces of the same `TinyBranchCNN`: positive input fires THEN, negative input fires ELSE.

In [ ]:
x_neg = torch.full((1, 1, 8, 8), -1.0)
trace_then = tl.trace(cnn, x_pos, vis_opt="none")
trace_else = tl.trace(cnn, x_neg, vis_opt="none")


def conditional_summary(trace_log):
    cond = trace_log.conditionals[0]
    return {
        "id": cond.id,
        "fired_arm_kind": cond.fired_arm_kind,
        "source_location": cond.source_location,
        "num_arms": cond.num_arms,
        "has_else": cond.has_else,
        "fired_arm_exec_ops": cond.fired_arm.execution_op_labels if cond.fired_arm else [],
    }


{
    "x_pos": conditional_summary(trace_then),
    "x_neg": conditional_summary(trace_else),
}

In [ ]:
then_path = OUT / "branch_then"
else_path = OUT / "branch_else"
trace_then.draw(
    vis_outpath=str(then_path), vis_save_only=True, vis_fileformat="svg", vis_mode="unrolled"
)
trace_else.draw(
    vis_outpath=str(else_path), vis_save_only=True, vis_fileformat="svg", vis_mode="unrolled"
)

print("Left: x.mean() > 0 fires THEN. Right: x.mean() < 0 fires ELSE.")
display(SVG(filename=str(then_path.with_suffix(".svg"))))

In [ ]:
display(SVG(filename=str(else_path.with_suffix(".svg"))))

## 5. Compare traces with a Bundle

A `Bundle` aligns multiple traces and lets you query them as one object. Layer labels become cross-trace handles; comparison helpers rank what changed.

In [ ]:
bundle = tl.bundle({"clean": clean, "zero_relu": zero_relu}, baseline="clean")

relu_view = bundle.layers["relu_1_2"]
{
    "members": list(bundle.members.keys()),
    "baseline": bundle.baseline_name,
    "super_layer_node_name": relu_view.node_name,
    "super_layer_op_type": relu_view.op_type,
    "super_layer_coverage": relu_view.coverage,
    "super_layer_member_shapes": {name: tuple(out.shape) for name, out in relu_view.outs.items()},
    "most_changed_top3": bundle.most_changed("clean", top_k=3),
}

In [ ]:
diff_path = OUT / "bundle_diff"
bundle.show_diff(vis_outpath=str(diff_path), vis_save_only=True)
display(SVG(filename=str(diff_path.with_suffix(".svg"))))

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [ ]:
bundle.show_diff(vis_outpath="out.pdf", format="pdf")

In [ ]:
shutil.rmtree(OUT, ignore_errors=True)
print("cleanup complete")